# Prerequisites

In [1]:
# get data for labs
!wget -nc -O around_the_world_in_80_days.txt https://www.gutenberg.org/ebooks/103.txt.utf-8

File ‘around_the_world_in_80_days.txt’ already there; not retrieving.


# 1. Word Count

Instructions:  
For each cell marked "double-click and add explanation here" please answer the question in your own words.  
In the section where you complete the code to perform basic nlp text cleaning and exploration tasks, the goal is to chain all of the transformations together in a single function. For learning and exploration purposes, it is acceptable to have each step seperate, but the last cell in this section should be one function with all transformations chained together.  
For steps c and f, it is acceptable to use your favorite chatbot to generate a list of common stop words (c) and punctuation (e) for use in the code. As these are common steps in nlp/text processing tasks, there are pleanty of libraries to help with this such as nltk, but there is no need to import extra dependencies for this lab unless you are already familiar with working with them.

In [2]:
# start a spark session and create spark context for making rdd
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("word_count") \
    .getOrCreate()

sc = spark.sparkContext

In [3]:
# Defind the rdd
rdd = sc.textFile('around_the_world_in_80_days.txt')

In [4]:
# view the first x lines of the rdd
rdd.take(20)

['The Project Gutenberg eBook of Around the World in Eighty Days',
 '    ',
 'This eBook is for the use of anyone anywhere in the United States and',
 'most other parts of the world at no cost and with almost no restrictions',
 'whatsoever. You may copy it, give it away or re-use it under the terms',
 'of the Project Gutenberg License included with this eBook or online',
 'at www.gutenberg.org. If you are not located in the United States,',
 'you will have to check the laws of the country where you are located',
 'before using this eBook.',
 '',
 'Title: Around the World in Eighty Days',
 '',
 'Author: Jules Verne',
 '',
 'Translator: George M. Towle',
 '',
 '',
 '        ',
 'Release date: January 1, 1994 [eBook #103]',
 '                Most recently updated: October 29, 2024']

In [5]:
# example lambda function
words = rdd.flatMap(lambda lines: lines.split(' '))

In [6]:
# Note and explain the output of the below command
words

PythonRDD[3] at RDD at PythonRDD.scala:59

`words` is not a list of words: it is an RDD object (`PythonRDD[...]`). `flatMap` is a **transformation**, and transformations are **lazy**: Spark only records the step, nothing is computed yet.

In [7]:
# Note and explain the output of the following command, focusing on the difference with the
# above command
words.collect()

['The',
 'Project',
 'Gutenberg',
 'eBook',
 'of',
 'Around',
 'the',
 'World',
 'in',
 'Eighty',
 'Days',
 '',
 '',
 '',
 '',
 '',
 'This',
 'eBook',
 'is',
 'for',
 'the',
 'use',
 'of',
 'anyone',
 'anywhere',
 'in',
 'the',
 'United',
 'States',
 'and',
 'most',
 'other',
 'parts',
 'of',
 'the',
 'world',
 'at',
 'no',
 'cost',
 'and',
 'with',
 'almost',
 'no',
 'restrictions',
 'whatsoever.',
 'You',
 'may',
 'copy',
 'it,',
 'give',
 'it',
 'away',
 'or',
 're-use',
 'it',
 'under',
 'the',
 'terms',
 'of',
 'the',
 'Project',
 'Gutenberg',
 'License',
 'included',
 'with',
 'this',
 'eBook',
 'or',
 'online',
 'at',
 'www.gutenberg.org.',
 'If',
 'you',
 'are',
 'not',
 'located',
 'in',
 'the',
 'United',
 'States,',
 'you',
 'will',
 'have',
 'to',
 'check',
 'the',
 'laws',
 'of',
 'the',
 'country',
 'where',
 'you',
 'are',
 'located',
 'before',
 'using',
 'this',
 'eBook.',
 '',
 'Title:',
 'Around',
 'the',
 'World',
 'in',
 'Eighty',
 'Days',
 '',
 'Author:',
 'Jules'

`collect()` is an **action**: Spark now reads the file, runs the `flatMap` and sends all the words back to the driver as a Python list. On a big dataset `collect()` should be avoided (everything goes into the driver memory), `take(n)` is safer.

In [8]:
# nicer print (only the first 20 words, the book has about 70,000)
for w in words.collect()[:20]:
    print(w)

The
Project
Gutenberg
eBook
of
Around
the
World
in
Eighty
Days





This
eBook
is
for


In [9]:
# Print first x words
words.take(20)

['The',
 'Project',
 'Gutenberg',
 'eBook',
 'of',
 'Around',
 'the',
 'World',
 'in',
 'Eighty',
 'Days',
 '',
 '',
 '',
 '',
 '',
 'This',
 'eBook',
 'is',
 'for']

In [10]:
%%time
# Use cell magic command to help understand what the rdd.flatMap function is doing in the next cell.
# Insert a text/markdown cell and explain in your own words.
print(rdd.map(lambda line: line.split(' ')).take(3))
print(rdd.flatMap(lambda line: line.split(' ')).take(3))
print(rdd.count(), "lines ->", words.count(), "words")

[['The', 'Project', 'Gutenberg', 'eBook', 'of', 'Around', 'the', 'World', 'in', 'Eighty', 'Days'], ['', '', '', '', ''], ['This', 'eBook', 'is', 'for', 'the', 'use', 'of', 'anyone', 'anywhere', 'in', 'the', 'United', 'States', 'and']]
['The', 'Project', 'Gutenberg']


8312 lines -> 68577 words
CPU times: user 10.6 ms, sys: 1.07 ms, total: 11.7 ms
Wall time: 318 ms


`map` returns **one element per line** (a list of words). `flatMap` **flattens** these lists: it returns **one element per word**. This is why the RDD goes from 8,312 lines to 68,577 words. `%%time` shows the time of the cell.

In [11]:
# Initialize a word counter by creating a tuple with word and cound of 1
words = rdd.flatMap(lambda lines: lines.split(' ')) \
                    .map(lambda word: (word, 1))

for w in words.collect()[:20]:
    print(w)

('The', 1)
('Project', 1)
('Gutenberg', 1)
('eBook', 1)
('of', 1)
('Around', 1)
('the', 1)
('World', 1)
('in', 1)
('Eighty', 1)
('Days', 1)
('', 1)
('', 1)
('', 1)
('', 1)
('', 1)
('This', 1)
('eBook', 1)
('is', 1)
('for', 1)


In [12]:
# a. count the occurence of each word
word_counts = words.reduceByKey(lambda a, b: a + b)

word_counts.take(10)

[('Gutenberg', 60),
 ('eBook', 6),
 ('of', 1875),
 ('Around', 4),
 ('', 2193),
 ('for', 407),
 ('use', 16),
 ('anyone', 6),
 ('United', 23),
 ('States', 10)]

In [13]:
# b. a common first step in text analysis, change all capital letters to lower case
lower_counts = words.map(lambda pair: (pair[0].lower(), pair[1])) \
                    .reduceByKey(lambda a, b: a + b)

lower_counts.take(10)

[('of', 1926),
 ('around', 32),
 ('world', 35),
 ('eighty', 27),
 ('days', 46),
 ('', 2193),
 ('this', 341),
 ('for', 414),
 ('use', 19),
 ('anyone', 6)]

In [14]:
# c. eliminate the stop words.
stop_words = {
    "a", "about", "after", "again", "all", "am", "an", "and", "any", "are",
    "as", "at", "be", "been", "before", "but", "by", "can", "could", "did",
    "do", "does", "for", "from", "had", "has", "have", "he", "her", "here",
    "him", "his", "how", "i", "if", "in", "into", "is", "it", "its", "me",
    "more", "my", "no", "not", "now", "of", "on", "one", "only", "or",
    "other", "our", "out", "over", "said", "she", "should", "so", "some",
    "than", "that", "the", "their", "them", "then", "there", "these",
    "they", "this", "those", "to", "too", "up", "upon", "very", "was", "we",
    "were", "what", "when", "where", "which", "while", "who", "whom", "why",
    "will", "with", "would", "you", "your"
}

no_stop_words = lower_counts.filter(lambda pair: pair[0] not in stop_words)

no_stop_words.take(10)

[('around', 32),
 ('world', 35),
 ('eighty', 27),
 ('days', 46),
 ('', 2193),
 ('use', 19),
 ('anyone', 6),
 ('united', 27),
 ('states', 14),
 ('most', 45)]

In [15]:
# d. sort in alphabetical order
no_stop_words.sortByKey().take(10)

[('', 2193),
 ('#103]', 1),
 ('#516,', 1),
 ('$5,000)', 1),
 ('&c.,', 1),
 ('($1', 1),
 ('(862)', 1),
 ('(a)', 1),
 ('(and', 1),
 ('(any', 1)]

In [16]:
# e. sort descending by word frequency
no_stop_words.sortBy(lambda pair: pair[1], ascending=False).take(10)

[('', 2193),
 ('mr.', 373),
 ('fogg', 365),
 ('phileas', 250),
 ('passepartout', 239),
 ('fogg,', 132),
 ('fix', 129),
 ('passepartout,', 121),
 ('“i', 115),
 ('two', 98)]

In [17]:
# f. remove punctuations and blank spaces
import re

clean_counts = no_stop_words \
    .map(lambda pair: (re.sub(r"[^\w]", "", pair[0]), pair[1])) \
    .filter(lambda pair: pair[0] != "") \
    .filter(lambda pair: pair[0] not in stop_words) \
    .reduceByKey(lambda a, b: a + b)

clean_counts.sortBy(lambda pair: pair[1], ascending=False).take(10)

[('fogg', 601),
 ('passepartout', 402),
 ('mr', 389),
 ('phileas', 255),
 ('fix', 240),
 ('aouda', 125),
 ('time', 125),
 ('himself', 124),
 ('train', 119),
 ('master', 104)]

`[^\w]` removes every character that is not a letter or a digit (punctuation, quotes like “ ”). Before this step, `fogg` and `fogg,` were counted as 2 different words, and the empty string `''` was the most frequent "word". The stop words are filtered again, because some were hidden by punctuation (`“i`, `and,`).

Final function: all the steps chained. Cleaning is done **before** `reduceByKey`, so only one shuffle is needed.

In [18]:
def word_count(lines, stop_words, alphabetical=False):
    counts = (lines
              .flatMap(lambda line: line.split(' '))
              .map(lambda word: re.sub(r"[^\w]", "", word.lower()))
              .filter(lambda word: word != "" and word not in stop_words)
              .map(lambda word: (word, 1))
              .reduceByKey(lambda a, b: a + b))
    if alphabetical:
        return counts.sortByKey()
    return counts.sortBy(lambda pair: pair[1], ascending=False)


word_count(rdd, stop_words).take(20)

[('fogg', 601),
 ('passepartout', 402),
 ('mr', 389),
 ('phileas', 255),
 ('fix', 240),
 ('aouda', 125),
 ('time', 125),
 ('himself', 124),
 ('train', 119),
 ('master', 104),
 ('two', 102),
 ('sir', 100),
 ('project', 99),
 ('hundred', 98),
 ('replied', 93),
 ('hours', 89),
 ('without', 88),
 ('thousand', 88),
 ('steamer', 88),
 ('well', 87)]

# 2. What does the following cell block do?
Comment the code below line by line after the provided hash-tag. You should be able to explain each line while respecting the pep8 style guide of 79 characters or less per line!

In [19]:
 # Create an RDD of tuples (name, age)
dataRDD = sc.parallelize([("Brooke", 20), ("Denny", 31), ("Jules", 30),
("TD", 35), ("Brooke", 25)])

# Try to undestand what this code does (line by line)
agesRDD = (dataRDD
  # (name, age) -> (name, (age, 1)): the 1 is used to count the records
  .map(lambda x: (x[0], (x[1], 1)))
  # same name: add the ages together and add the counters together
  .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))
  # (name, (total_age, count)) -> (name, total_age / count) = average age
  .map(lambda x: (x[0], x[1][0]/x[1][1])))

agesRDD.collect()

[('Brooke', 22.5), ('Jules', 30.0), ('Denny', 31.0), ('TD', 35.0)]

The code computes the **average age per name**. Brooke appears twice (20 and 25), so her average is 22.5.

## 3. Function timing.

- write a simple python timer function for seeing how quickly your rdd runs as written. change the order of the steps in order to make the rdd run as optimally as possible


In [20]:
import time


def timer(pipeline, repeat=3):
    # run the pipeline several times and return the average time
    times = []
    for _ in range(repeat):
        start = time.perf_counter()
        pipeline().take(20)  # an action is needed, RDDs are lazy
        times.append(time.perf_counter() - start)
    return sum(times) / repeat

In [21]:
def pipeline_as_written():
    # order of the exercise: count first, clean after (3 shuffles)
    return (rdd
            .flatMap(lambda line: line.split(' '))
            .map(lambda word: (word, 1))
            .reduceByKey(lambda a, b: a + b)
            .map(lambda pair: (pair[0].lower(), pair[1]))
            .reduceByKey(lambda a, b: a + b)
            .map(lambda pair: (re.sub(r"[^\w]", "", pair[0]), pair[1]))
            .reduceByKey(lambda a, b: a + b)
            .filter(lambda pair: pair[0] != "")
            .filter(lambda pair: pair[0] not in stop_words)
            .sortBy(lambda pair: pair[1], ascending=False))


def pipeline_optimized():
    # clean and filter first, then count (1 shuffle)
    return word_count(rdd, stop_words)


print("as written:", round(timer(pipeline_as_written), 3), "s")
print("optimized :", round(timer(pipeline_optimized), 3), "s")
print("same result:", set(pipeline_as_written().collect())
      == set(pipeline_optimized().collect()))

as written: 0.481 s


optimized : 0.315 s


same result: True


The optimized order is faster because:
- `map` and `filter` are **narrow** transformations (no data exchange), `reduceByKey` is **wide** (shuffle). Cleaning before counting needs only 1 `reduceByKey` instead of 3.
- Filtering the stop words early means less data to shuffle.

The results are the same, only the order of the steps changes.

## 4. Text Comparison

- perform eda on the original french version of the [book](https://www.gutenberg.org/ebooks/46541.txt.utf-8) and compare the two

In [22]:
!wget -nc -O le_tour_du_monde_en_80_jours.txt https://www.gutenberg.org/ebooks/46541.txt.utf-8

File ‘le_tour_du_monde_en_80_jours.txt’ already there; not retrieving.


In [23]:
def remove_gutenberg_text(lines):
    # keep only the book, between the "*** START" and "*** END" lines
    indexed = lines.zipWithIndex()
    start = indexed.filter(lambda x: x[0].startswith("*** START")).first()[1]
    end = indexed.filter(lambda x: x[0].startswith("*** END")).first()[1]
    return indexed.filter(lambda x: start < x[1] < end).map(lambda x: x[0])


rdd_en = remove_gutenberg_text(rdd)
rdd_fr = remove_gutenberg_text(
    sc.textFile('le_tour_du_monde_en_80_jours.txt'))

stop_words_fr = {
    "a", "à", "ai", "au", "aux", "avait", "avec", "c", "ce", "cela", "ces",
    "cet", "cette", "comme", "d", "dans", "de", "des", "dit", "donc", "du",
    "elle", "en", "est", "et", "été", "était", "être", "il", "ils", "j",
    "je", "l", "la", "le", "les", "leur", "lui", "m", "mais", "me", "même",
    "mon", "n", "ne", "nous", "on", "ou", "où", "par", "pas", "pour",
    "qu", "que", "qui", "s", "sa", "sans", "se", "ses", "si", "son", "sont",
    "sur", "t", "tout", "un", "une", "vous", "y", "plus", "bien", "très",
    "ont", "fait", "avoir", "dont", "alors", "encore"
}

# in French the apostrophe joins 2 words (l'homme), so we split on it too
rdd_fr = rdd_fr.map(lambda line: line.replace("'", " ").replace("’", " "))

counts_en = word_count(rdd_en, stop_words)
counts_fr = word_count(rdd_fr, stop_words_fr)

In [24]:
def total_words(lines):
    return lines.flatMap(lambda line: line.split()).count()


print("lines          EN:", rdd_en.count(), "| FR:", rdd_fr.count())
print("total words    EN:", total_words(rdd_en), "| FR:", total_words(rdd_fr))
print("distinct words EN:", counts_en.count(), "| FR:", counts_fr.count())

lines          EN: 7934 | FR: 9585
total words    EN: 63334 | FR: 71915


distinct words EN: 7069 | FR: 9379


In [25]:
print("Top 10 EN:", counts_en.take(10))
print("Top 10 FR:", counts_fr.take(10))

Top 10 EN: [('fogg', 601), ('passepartout', 402), ('mr', 389), ('phileas', 255), ('fix', 239), ('aouda', 125), ('time', 125), ('himself', 124), ('train', 119), ('master', 104)]
Top 10 FR: [('fogg', 688), ('passepartout', 453), ('phileas', 331), ('mr', 287), ('fix', 285), ('heures', 242), ('répondit', 215), ('deux', 147), ('monsieur', 145), ('aouda', 135)]


In [26]:
# names are the same in both versions: join on the word
names = ["fogg", "passepartout", "phileas", "fix", "aouda"]

counts_en.join(counts_fr) \
    .filter(lambda pair: pair[0] in names) \
    .collect()

[('fogg', (601, 688)),
 ('passepartout', (402, 453)),
 ('phileas', (255, 331)),
 ('aouda', (125, 135)),
 ('fix', (239, 285))]


- The French version is longer: 71,915 words vs 63,334, and has more distinct words (9,379 vs 7,069), because French has more forms for the same word (conjugations, feminine, plural).
- The most frequent words are the same characters in both versions: Fogg, Passepartout, Phileas, Fix, Aouda.
- The names appear more often in French (Fogg: 688 vs 601), the English translation probably replaces some of them with "he".
- `mr` is also in the French top words: Verne uses the English title "Mr." in the original.